<a href="https://colab.research.google.com/github/Krisnaaassss/heart-disease-classification-soft-voting/blob/main/Resampling_Dataset_Long_Beach_VA_and_Heart_Failure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Setup and Configuration


## 1.1 Library Installation and Imports


In [ ]:
# CELL 1 — Install and Import
!pip install xgboost lightgbm imbalanced-learn scikit-optimize -q

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import pandas as pd
import numpy as np
from collections import Counter

from sklearn.model_selection import (StratifiedKFold, cross_validate,
                                      GridSearchCV, RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (make_scorer, precision_score, recall_score, f1_score,
                             accuracy_score, roc_auc_score)
from sklearn.ensemble import VotingClassifier
from sklearn.base import BaseEstimator, ClassifierMixin, clone

from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb
import lightgbm as lgb

from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline

from skopt import BayesSearchCV, gp_minimize
from skopt.space import Real, Integer, Categorical

print('Libraries loaded successfully.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.5 MB/s eta 0:00:00
Libraries loaded successfully.


## 1.2 Global Configuration


In [ ]:
# CELL 2 — Global Configuration

RANDOM_STATE = 42
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 5


SCORING = {
    'accuracy':  'accuracy',
    'precision': make_scorer(precision_score, zero_division=0),
    'recall':    make_scorer(recall_score,    zero_division=0),
    'f1':        make_scorer(f1_score,        zero_division=0),
    'roc_auc':   'roc_auc',
}


RESAMPLING_METHODS = [
    'none',
    'smote',
    'borderline_smote',
    'smote_tomek',
]

def get_resampler(resampling_method):
    if resampling_method == 'none':
        return None
    if resampling_method == 'smote':
        return SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
    if resampling_method == 'borderline_smote':
        return BorderlineSMOTE(
            random_state=RANDOM_STATE,
            kind='borderline-1',
            k_neighbors=5,
            m_neighbors=10,
        )
    if resampling_method == 'smote_tomek':
        return SMOTETomek(
            random_state=RANDOM_STATE,
            smote=SMOTE(random_state=RANDOM_STATE, k_neighbors=5),
        )
    raise ValueError(
        "resampling_method must be one of: 'none', 'smote', "
        "'borderline_smote', or 'smote_tomek'."
    )

def get_resampling_label(resampling_method):
    labels = {
        'none': 'No Resampling',
        'smote': 'SMOTE',
        'borderline_smote': 'Borderline-SMOTE',
        'smote_tomek': 'SMOTE-Tomek',
    }
    return labels[resampling_method]

print('Global configuration completed.')

Global configuration completed.


# 2. Main Evaluation Pipeline


## 2.1 Bayesian Optimization for Baseline Models


In [ ]:
# CELL 4 — Baseline Models with Bayesian Optimization

def _build_models_and_spaces():
    return {
        'SVM': (
            SVC(probability=True, random_state=RANDOM_STATE),
            {'model__C':      Real(1e-2, 1e2, prior='log-uniform'),
             'model__kernel': Categorical(['rbf', 'linear'])},
        ),
        'Naive Bayes': (
            GaussianNB(),
            {'model__var_smoothing': Real(1e-10, 1e-6, prior='log-uniform')},
        ),
        'Random Forest': (
            RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
            {'model__n_estimators': Integer(50, 300),
             'model__max_depth':    Integer(3, 30)},
        ),
        'J48 (DecTree)': (
            DecisionTreeClassifier(random_state=RANDOM_STATE),
            {'model__max_depth':  Integer(2, 20),
             'model__criterion':  Categorical(['gini', 'entropy'])},
        ),
        'Logistic Regression': (
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            {'model__C': Real(1e-2, 1e2, prior='log-uniform')},
        ),
        'XGBoost': (
            xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', n_jobs=1),
            {'model__n_estimators':  Integer(50, 300),
             'model__max_depth':     Integer(2, 10),
             'model__learning_rate': Real(1e-2, 3e-1, prior='log-uniform')},
        ),
        'LightGBM': (
            lgb.LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, n_jobs=1),
            {'model__n_estimators':  Integer(50, 500),
             'model__num_leaves':    Integer(7, 150),
             'model__learning_rate': Real(1e-2, 3e-1, prior='log-uniform')},
        ),
        'KNN': (
            KNeighborsClassifier(n_jobs=1),
            {'model__n_neighbors': Integer(3, 15),
             'model__weights':     Categorical(['uniform', 'distance'])},
        ),
    }


def _make_pipe(model, resampling_method='none'):
    resampler = get_resampler(resampling_method)

    if resampler is not None:
        return ImbPipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('resampler', resampler),
            ('scaler', StandardScaler()),
            ('model', model),
        ])

    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', model),
    ])


def run_nested_cv_baseline(X, y, dataset_name='', resampling_method='none', n_bayes_iter=20):
    outer_cv = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    models = _build_models_and_spaces()

    print(f'\n  Nested 5-Fold CV — Baseline [Bayesian Optimization] [{dataset_name}] | {get_resampling_label(resampling_method)}')
    print(f'  {"Model":<22} {"Acc":>8} {"±":>5} {"Prec":>8} {"Rec":>8} {"F1":>8} {"AUC":>8}')
    print('  ' + '─' * 72)

    results = {}
    for name, (model, space) in models.items():
        pipe = _make_pipe(model, resampling_method)

        bs = BayesSearchCV(
            pipe, search_spaces=space, n_iter=n_bayes_iter,
            cv=inner_cv, scoring='f1', n_jobs=1, refit=True,
            random_state=RANDOM_STATE,
        )

        score = cross_validate(bs, X, y, cv=outer_cv, scoring=SCORING,
                               n_jobs=-1, return_train_score=False)

        acc  = score['test_accuracy']
        prec = score['test_precision']
        rec  = score['test_recall']
        f1v  = score['test_f1']
        auc  = score['test_roc_auc']
        results[name] = {'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1v, 'auc': auc}

        print(f'  {name:<22}'
              f' {acc.mean()*100:>7.2f}%'
              f' ±{acc.std()*100:.2f}'
              f' {prec.mean()*100:>7.2f}%'
              f' {rec.mean()*100:>7.2f}%'
              f' {f1v.mean()*100:>7.2f}%'
              f' {auc.mean()*100:>7.2f}%')

    return results


def get_best_params_for_voting(X, y, dataset_name='', resampling_method='none', n_bayes_iter=25):
    inner_cv = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    voting_members = {
        'svm': (
            SVC(probability=True, random_state=RANDOM_STATE),
            {'model__C':      Real(1e-2, 1e2, prior='log-uniform'),
             'model__kernel': Categorical(['rbf', 'linear'])},
        ),
        'rf': (
            RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
            {'model__n_estimators': Integer(50, 300),
             'model__max_depth':    Integer(3, 30)},
        ),
        'lgbm': (
            lgb.LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, n_jobs=1),
            {'model__n_estimators':  Integer(50, 500),
             'model__num_leaves':    Integer(7, 150),
             'model__learning_rate': Real(1e-2, 3e-1, prior='log-uniform')},
        ),
    }

    print(f'\n  Bayesian optimization for Soft Voting base learners [{dataset_name}]')

    best_params = {}
    for key, (model, space) in voting_members.items():
        pipe = _make_pipe(model, resampling_method)
        bs = BayesSearchCV(
            pipe, search_spaces=space, n_iter=n_bayes_iter,
            cv=inner_cv, scoring='f1', n_jobs=-1, refit=True,
            random_state=RANDOM_STATE,
        )
        bs.fit(X, y)
        clean_params = {k.replace('model__', ''): v for k, v in bs.best_params_.items()}
        best_params[key] = clean_params
        print(f'    {key:<6}: {clean_params}  (F1 inner CV = {bs.best_score_*100:.2f}%)')

    return best_params

print('Baseline evaluation functions initialized.')


Baseline evaluation functions initialized.


## 2.2 Soft Voting Ensemble


In [ ]:
# CELL 5 — Soft Voting Ensemble


class WeightedSoftVoting(BaseEstimator, ClassifierMixin):

    def __init__(self, svm=None, rf=None, lgbm=None,
                 w_svm=1.0, w_rf=1.0, w_lgbm=1.0):
        self.svm = svm
        self.rf = rf
        self.lgbm = lgbm
        self.w_svm = w_svm
        self.w_rf = w_rf
        self.w_lgbm = w_lgbm

    def fit(self, X, y):
        self.svm_  = clone(self.svm).fit(X, y)
        self.rf_   = clone(self.rf).fit(X, y)
        self.lgbm_ = clone(self.lgbm).fit(X, y)
        self.classes_ = self.svm_.classes_
        return self

    def predict_proba(self, X):
        w = np.array([self.w_svm, self.w_rf, self.w_lgbm], dtype=float)
        w = w / w.sum()
        proba = (w[0] * self.svm_.predict_proba(X)
                 + w[1] * self.rf_.predict_proba(X)
                 + w[2] * self.lgbm_.predict_proba(X))
        return proba

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]


def _get_oof_base_proba(X_tr, y_tr, best_params, resampling_method, inner_cv):
    X_tr = X_tr.reset_index(drop=True)
    y_tr = pd.Series(np.asarray(y_tr)).reset_index(drop=True)
    n = len(X_tr)

    oof_svm  = np.zeros(n)
    oof_rf   = np.zeros(n)
    oof_lgbm = np.zeros(n)

    for in_idx, val_idx in inner_cv.split(X_tr, y_tr):
        X_in, X_val = X_tr.iloc[in_idx], X_tr.iloc[val_idx]
        y_in, y_val = y_tr.iloc[in_idx], y_tr.iloc[val_idx]

        imputer = SimpleImputer(strategy='median')
        X_in_imp  = imputer.fit_transform(X_in)
        X_val_imp = imputer.transform(X_val)

        X_in_res, y_in_res = X_in_imp, y_in
        resampler = get_resampler(resampling_method)
        if resampler is not None:
            X_in_res, y_in_res = resampler.fit_resample(X_in_imp, y_in)

        scaler = StandardScaler()
        X_in_scaled  = scaler.fit_transform(X_in_res)
        X_val_scaled = scaler.transform(X_val_imp)

        svm  = SVC(probability=True, random_state=RANDOM_STATE,
                   **best_params['svm']).fit(X_in_scaled, y_in_res)
        rf   = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1,
                   **best_params['rf']).fit(X_in_scaled, y_in_res)
        lgbm = lgb.LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, n_jobs=1,
                   **best_params['lgbm']).fit(X_in_scaled, y_in_res)

        oof_svm[val_idx]  = svm.predict_proba(X_val_scaled)[:, 1]
        oof_rf[val_idx]   = rf.predict_proba(X_val_scaled)[:, 1]
        oof_lgbm[val_idx] = lgbm.predict_proba(X_val_scaled)[:, 1]

    return oof_svm, oof_rf, oof_lgbm, y_tr.values


def _search_best_weights(oof_svm, oof_rf, oof_lgbm, y_true, n_calls=15):
    space = [
        Real(0.1, 3.0, name='w_svm'),
        Real(0.1, 3.0, name='w_rf'),
        Real(0.1, 3.0, name='w_lgbm'),
    ]

    def objective(params):
        w_svm, w_rf, w_lgbm = params
        w = np.array([w_svm, w_rf, w_lgbm])
        w = w / w.sum()
        proba = w[0] * oof_svm + w[1] * oof_rf + w[2] * oof_lgbm
        pred = (proba >= 0.5).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        return -f1  # gp_minimize meminimalkan -> kita minimalkan negatif F1

    res = gp_minimize(
        objective, space, n_calls=n_calls, n_initial_points=min(5, n_calls),
        random_state=RANDOM_STATE, verbose=False,
    )

    best_weights = {'w_svm': res.x[0], 'w_rf': res.x[1], 'w_lgbm': res.x[2]}
    best_f1_inner = -res.fun
    return best_weights, best_f1_inner


def _fit_final_and_predict(X_tr, y_tr, X_te, best_params, best_weights, resampling_method):
    svm_fixed  = SVC(probability=True, random_state=RANDOM_STATE, **best_params['svm'])
    rf_fixed   = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1, **best_params['rf'])
    lgbm_fixed = lgb.LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, n_jobs=1, **best_params['lgbm'])

    voting_model = WeightedSoftVoting(
        svm=svm_fixed, rf=rf_fixed, lgbm=lgbm_fixed,
        w_svm=best_weights['w_svm'], w_rf=best_weights['w_rf'], w_lgbm=best_weights['w_lgbm'],
    )
    pipe = _make_pipe(voting_model, resampling_method)
    pipe.fit(X_tr, y_tr)

    y_pred  = pipe.predict(X_te)
    y_proba = pipe.predict_proba(X_te)[:, 1]
    return y_pred, y_proba


def run_nested_cv_soft_voting(X, y, dataset_name='', resampling_method='none',
                              n_bayes_iter_params=20, n_bayes_iter_weights=15):
    X = X.reset_index(drop=True)
    y = pd.Series(np.asarray(y)).reset_index(drop=True)

    outer_cv = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    accs, precs, recs, f1s, aucs = [], [], [], [], []
    fold_info = []

    print(f'\n  Nested 5-Fold CV — Soft Voting [Bayesian Optimization] [{dataset_name}] | {get_resampling_label(resampling_method)}')

    for fold_idx, (tr_idx, te_idx) in enumerate(outer_cv.split(X, y), start=1):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

        best_params = get_best_params_for_voting(
            X_tr, y_tr, dataset_name=f'{dataset_name} | fold {fold_idx}',
            resampling_method=resampling_method, n_bayes_iter=n_bayes_iter_params,
        )

        oof_svm, oof_rf, oof_lgbm, y_oof = _get_oof_base_proba(
            X_tr, y_tr, best_params, resampling_method, inner_cv,
        )

        best_weights, best_f1_inner = _search_best_weights(
            oof_svm, oof_rf, oof_lgbm, y_oof, n_calls=n_bayes_iter_weights,
        )

        y_pred, y_proba = _fit_final_and_predict(
            X_tr, y_tr, X_te, best_params, best_weights, resampling_method,
        )

        acc  = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, zero_division=0)
        rec  = recall_score(y_te, y_pred, zero_division=0)
        f1v  = f1_score(y_te, y_pred, zero_division=0)
        try:
            auc = roc_auc_score(y_te, y_proba)
        except ValueError:
            auc = np.nan

        accs.append(acc); precs.append(prec); recs.append(rec)
        f1s.append(f1v); aucs.append(auc)
        fold_info.append({'base_params': best_params, 'weights': best_weights,
                          'f1_inner_oof': best_f1_inner})

        print(f'    Fold {fold_idx}: Acc={acc*100:.2f}% Prec={prec*100:.2f}% '
              f'Rec={rec*100:.2f}% F1={f1v*100:.2f}% AUC={auc*100:.2f}%  '
              f'| weights={ {k: round(v,3) for k,v in best_weights.items()} }')

    acc  = np.array(accs); prec = np.array(precs); rec = np.array(recs)
    f1v  = np.array(f1s);  auc  = np.array(aucs)

    label = f'Soft Voting | {get_resampling_label(resampling_method)}'
    print(f'\n  {label} — mean across 5 folds')
    print(f'  {"Soft Voting Ensemble":<22}'
          f' {acc.mean()*100:>7.2f}%'
          f' ±{acc.std()*100:.2f}'
          f' {prec.mean()*100:>7.2f}%'
          f' {rec.mean()*100:>7.2f}%'
          f' {f1v.mean()*100:>7.2f}%'
          f' {auc.mean()*100:>7.2f}%')

    return {'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1v, 'auc': auc,
            'fold_info': fold_info}

print('Soft Voting Ensemble functions initialized.')


Soft Voting Ensemble functions initialized.


## 2.3 Dataset Evaluation Pipeline


In [ ]:
# CELL 7 — Dataset Evaluation Pipeline

def evaluate_dataset(df, target_col, dataset_name,
                     binarize=True,
                     compare_resampling=False):
    print('\n' + '=' * 72)
    print(f'  DATASET: {dataset_name}')
    print('=' * 72)

    df = df.dropna(subset=[target_col]).copy()
    if binarize:
        df[target_col] = (pd.to_numeric(df[target_col], errors='coerce') > 0).astype(int)

    X_all = df.drop(columns=[target_col])
    y = df[target_col].astype(int)

    counter = Counter(y)
    majority = max(counter.values())
    minority = min(counter.values())
    imbalance_ratio = majority / minority

    print(f'  Records          : {len(df)}')
    print(f'  Initial features : {X_all.shape[1]}')
    print(f'  Class distribution: {dict(counter)}')
    print(f'  Imbalance ratio   : {imbalance_ratio:.2f}:1')

    X_sel = X_all.copy()
    selected = X_all.columns.tolist()

    methods_to_run = RESAMPLING_METHODS if compare_resampling else ['none']

    experiments = {}

    for resampling_method in methods_to_run:
        resampling_label = get_resampling_label(resampling_method)

        print('\n' + '─' * 72)
        print(f'  RESAMPLING METHOD: {resampling_label}')
        print('─' * 72)

        base_res = run_nested_cv_baseline(
            X_sel,
            y,
            dataset_name=dataset_name,
            resampling_method=resampling_method,
        )

        sv_res = run_nested_cv_soft_voting(
            X_sel,
            y,
            dataset_name=dataset_name,
            resampling_method=resampling_method,
        )
        base_res['Soft Voting Ensemble'] = sv_res

        experiments[resampling_label] = {
            'resampling_method': resampling_method,
            'results': base_res,
        }

    primary_method = 'none'
    primary_label = get_resampling_label(primary_method)
    primary_experiment = experiments[primary_label]

    return {
        'dataset': dataset_name,
        'n_records': len(df),
        'n_features_total': X_all.shape[1],
        'n_features_selected': X_all.shape[1],
        'selected_features': selected,
        'class_dist': dict(counter),
        'imbalance_ratio': imbalance_ratio,
        'results': primary_experiment['results'],
        'experiments': experiments,
    }

print('Evaluation pipeline initialized.')

Evaluation pipeline initialized.


# 3. Resampling Experiments


## 3.1 Long Beach VA


In [ ]:
# CELL 10 — Long Beach VA

url_va = ('https://archive.ics.uci.edu/ml/machine-learning-databases/'
          'heart-disease/processed.va.data')

cols = ['age','sex','cp','trestbps','chol','fbs','restecg',
        'thalach','exang','oldpeak','slope','ca','thal','target']

df_va = pd.read_csv(url_va, names=cols, na_values='?')
for c in cols:
    df_va[c] = pd.to_numeric(df_va[c], errors='coerce')

miss_rate = df_va.isnull().mean()
drop_cols = miss_rate[miss_rate > 0.9].index.tolist()
if drop_cols:
    print(f'Columns removed (>90% missing values): {drop_cols}')
    df_va = df_va.drop(columns=drop_cols)

summary_va = evaluate_dataset(
    df_va, 'target', 'Long Beach VA',
    compare_resampling=True
)

Columns removed (>90% missing values): ['ca']

  DATASET: Long Beach VA
  Records       : 200
  Initial features   : 12
  Class distribution: {1: 149, 0: 51}
  Imbalance ratio    : 2.92:1

────────────────────────────────────────────────────────────────────────
  EKSPERIMEN RESAMPLING: No Resampling
────────────────────────────────────────────────────────────────────────

  Nested 5-Fold CV — Baseline [Bayesian Optimization] [Long Beach VA] | No Resampling
  Model                       Acc     ±     Prec      Rec       F1      AUC
  ────────────────────────────────────────────────────────────────────────
  SVM                      75.50% ±2.45   75.29%  100.00%   85.89%   69.87%
  Naive Bayes              74.00% ±6.04   79.21%   88.55%   83.50%   70.99%
  Random Forest            74.50% ±1.00   75.03%   98.67%   85.21%   69.47%
  J48 (DecTree)            75.00% ±2.74   77.40%   93.95%   84.85%   56.59%
  Logistic Regression      74.50% ±1.00   74.76%   99.33%   85.30%   70.04%
  XGBoos

## 3.2 Heart Failure Clinical Records


In [ ]:
# CELL 12 — Heart Failure Clinical Records

url_hf = ('https://archive.ics.uci.edu/ml/machine-learning-databases/'
          '00519/heart_failure_clinical_records_dataset.csv')

df_hf = pd.read_csv(url_hf)
print('Dataset shape:', df_hf.shape)
print(f'DEATH_EVENT distribution: {Counter(df_hf["DEATH_EVENT"])}')

summary_hf = evaluate_dataset(
    df_hf, 'DEATH_EVENT', 'Heart Failure Clinical',
    binarize=False,
    compare_resampling=True
)

Dataset shape: (299, 13)
DEATH_EVENT distribution: Counter({0: 203, 1: 96})

  DATASET: Heart Failure Clinical
  Records       : 299
  Initial features   : 12
  Class distribution: {1: 96, 0: 203}
  Imbalance ratio    : 2.11:1

────────────────────────────────────────────────────────────────────────
  EKSPERIMEN RESAMPLING: No Resampling
────────────────────────────────────────────────────────────────────────

  Nested 5-Fold CV — Baseline [Bayesian Optimization] [Heart Failure Clinical] | No Resampling
  Model                       Acc     ±     Prec      Rec       F1      AUC
  ────────────────────────────────────────────────────────────────────────
  SVM                      82.26% ±2.61   78.13%   62.42%   69.22%   86.64%
  Naive Bayes              77.59% ±3.29   74.43%   46.89%   57.45%   85.15%
  Random Forest            83.26% ±4.56   76.64%   68.68%   72.13%   89.82%
  J48 (DecTree)            80.94% ±2.48   73.31%   65.42%   68.05%   85.44%
  Logistic Regression      83.59% ±4

# 4. Results Summary


## 4.1 F1-Score Comparison


In [ ]:
# CELL 15 — F1-Score Comparison

all_summaries = [
    summary_va,
    summary_hf,
]

model_order = ['SVM', 'Naive Bayes', 'Random Forest', 'J48 (DecTree)',
               'Logistic Regression', 'XGBoost', 'LightGBM', 'KNN',
               'Soft Voting Ensemble']

# F1 Table
f1_table = pd.DataFrame(index=model_order,
                        columns=[s['dataset'] for s in all_summaries],
                        dtype=float)
for s in all_summaries:
    for m in model_order:
        if m in s['results']:
            f1_table.loc[m, s['dataset']] = s['results'][m]['f1'].mean() * 100

print('Mean F1-Score (%) — Nested 5-Fold CV\n')
print(f1_table.round(2).to_string())

print('\n\nBest model per dataset based on F1-Score:')
for ds in f1_table.columns:
    best_m = f1_table[ds].idxmax()
    best_v = f1_table[ds].max()
    sv_v   = f1_table.loc['Soft Voting Ensemble', ds]
    delta  = sv_v - best_v if best_m != 'Soft Voting Ensemble' else 0
    flag   = f'  (Soft Voting: {sv_v:.2f}%, difference {delta:+.2f}%)' if best_m != 'Soft Voting Ensemble' else ''
    print(f'  {ds:<25}: {best_m:<22} = {best_v:.2f}%{flag}')

Mean F1-Score (%) — Nested 5-Fold CV

                      Hungarian  Long Beach VA  Statlog Heart  Heart Failure Clinical  Cleveland
SVM                       76.17          80.37          82.29                   69.58      82.37
Naive Bayes               71.87          79.23          81.54                   65.86      83.85
Random Forest             75.34          83.65          77.95                   74.58      81.43
J48 (DecTree)             70.17          82.16          68.75                   72.12      75.48
Logistic Regression       76.15          78.08          81.03                   70.58      81.78
XGBoost                   71.11          82.47          77.35                   72.30      79.87
LightGBM                  71.88          79.89          79.40                   73.91      79.60
KNN                       74.01          76.46          79.54                   59.35      80.59
Soft Voting Ensemble      76.36          83.33          80.77                   73.02    

## 4.2 Accuracy Comparison


In [ ]:
# CELL 16 — Accuracy Comparison

acc_table = pd.DataFrame(index=model_order,
                         columns=[s['dataset'] for s in all_summaries],
                         dtype=float)
for s in all_summaries:
    for m in model_order:
        if m in s['results']:
            acc_table.loc[m, s['dataset']] = s['results'][m]['acc'].mean() * 100

print('Mean Accuracy (%) — Nested 5-Fold CV\n')
print(acc_table.round(2).to_string())

Mean Accuracy (%) — Nested 5-Fold CV

                      Hungarian  Long Beach VA  Statlog Heart  Heart Failure Clinical  Cleveland
SVM                       84.37           71.0          85.19                   78.60      84.48
Naive Bayes               75.55           70.5          84.07                   78.93      85.46
Random Forest             83.02           74.5          81.48                   83.94      84.15
J48 (DecTree)             79.25           73.0          74.07                   81.26      77.85
Logistic Regression       84.03           70.0          84.07                   79.28      84.14
XGBoost                   80.29           73.0          81.11                   81.92      81.83
LightGBM                  80.97           70.0          82.22                   82.94      82.17
KNN                       82.99           67.5          82.59                   70.90      82.83
Soft Voting Ensemble      84.38           74.5          83.70                   82.27    

## 4.3 Resampling Method Comparison


In [ ]:
# CELL 17 — Resampling Method Comparison

def create_resampling_comparison_table(summaries):
    rows = []

    for summary in summaries:
        for resampling_label, experiment in summary.get('experiments', {}).items():
            for model_name, metrics in experiment['results'].items():
                rows.append({
                    'Dataset': summary['dataset'],
                    'Resampling Method': resampling_label,
                    'Model': model_name,
                    'Accuracy (%)': metrics['acc'].mean() * 100,
                    'Precision (%)': metrics['prec'].mean() * 100,
                    'Recall (%)': metrics['rec'].mean() * 100,
                    'F1-Score (%)': metrics['f1'].mean() * 100,
                    'AUC (%)': metrics['auc'].mean() * 100,
                    'F1 Std (%)': metrics['f1'].std() * 100,
                })

    return pd.DataFrame(rows)

resampling_comparison_table = create_resampling_comparison_table([
    summary_va,
    summary_hf,
])

print('Results for all models across resampling methods\n')
print(resampling_comparison_table.round(2).to_string(index=False))


soft_voting_resampling_table = (
    resampling_comparison_table[
        resampling_comparison_table['Model'] == 'Soft Voting Ensemble'
    ]
    .sort_values(
        by=['Dataset', 'F1-Score (%)'],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

print('\n\nResampling comparison — Soft Voting Ensemble\n')
print(soft_voting_resampling_table.round(2).to_string(index=False))


best_resampling_per_dataset = (
    soft_voting_resampling_table
    .sort_values(by=['Dataset', 'F1-Score (%)'], ascending=[True, False])
    .groupby('Dataset', as_index=False)
    .first()
)

print('\n\nSelected resampling method — Soft Voting Ensemble\n')

selected_rows = []

hf_rows = soft_voting_resampling_table[
    soft_voting_resampling_table['Dataset'] == 'Heart Failure Clinical'
]
if not hf_rows.empty:
    selected_rows.append(
        hf_rows.sort_values('Accuracy (%)', ascending=False).head(1)
    )

lb_rows = soft_voting_resampling_table[
    soft_voting_resampling_table['Dataset'] == 'Long Beach VA'
]
if not lb_rows.empty:
    selected_rows.append(
        lb_rows.sort_values('F1-Score (%)', ascending=False).head(1)
    )

selected_resampling = pd.concat(selected_rows, ignore_index=True)

print(selected_resampling.round(2).to_string(index=False))


Results for all models across resampling methods

               Dataset Resampling Method                Model  Accuracy (%)  Precision (%)  Recall (%)  F1-Score (%)  AUC (%)  F1 Std (%)
         Long Beach VA     No Resampling                  SVM         75.50          75.29      100.00         85.89    69.87        1.34
         Long Beach VA     No Resampling          Naive Bayes         74.00          79.21       88.55         83.50    70.99        3.96
         Long Beach VA     No Resampling        Random Forest         74.50          75.03       98.67         85.21    69.47        0.61
         Long Beach VA     No Resampling        J48 (DecTree)         75.00          77.40       93.95         84.85    56.59        1.62
         Long Beach VA     No Resampling  Logistic Regression         74.50          74.76       99.33         85.30    70.04        0.64
         Long Beach VA     No Resampling              XGBoost         74.50          74.76       99.33         85.30    67